<a href="https://colab.research.google.com/github/salia40/mini-project-salia/blob/main/skincare%20(crud)%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. HUBUNGKAN KE GOOGLE DRIVE (TARUH PALING ATAS)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2. INSTALL & IMPORT LIBRARY
# ==========================================
!pip install gradio pandas

import gradio as gr
import pandas as pd
import json
import os

# ==========================================
# 3. UBAH LOKASI DATABASE KE GOOGLE DRIVE
# ==========================================
# File database akan tersimpan otomatis di dalam Google Drive Anda
DB_FILE = "/content/drive/MyDrive/database_skincare.json"

# Data awal jika file di Drive belum pernah dibuat
DATA_AWAL = [
    {
        "ID": 1,
        "Nama Produk": "Hydra Glow Serum",
        "Brand": "Wardah",
        "Kategori": "Serum",
        "Tipe Kulit": "Kering",
        "Harga": 85000,
        "Stok": 15,
        "Deskripsi": "Serum pelembab dengan Hyaluronic Acid."
    },
    {
        "ID": 2,
        "Nama Produk": "Clean & Fresh Facial Wash",
        "Brand": "Purbasari",
        "Kategori": "Cleanser",
        "Tipe Kulit": "Berminyak",
        "Harga": 35000,
        "Stok": 30,
        "Deskripsi": "Pembersih wajah untuk mengontrol minyak berlebih."
    }
]

# --- FUNGSI DATABASE (LOAD & SAVE PERMANEN) ---
def muat_data():
    """Membaca data dari file JSON di Google Drive."""
    if os.path.exists(DB_FILE):
        try:
            with open(DB_FILE, "r") as f:
                return json.load(f)
        except:
            return DATA_AWAL
    else:
        # Buat file awal jika belum ada
        simpan_data(DATA_AWAL)
        return DATA_AWAL

def simpan_data(data):
    """Menyimpan data langsung ke Google Drive secara permanen."""
    with open(DB_FILE, "w") as f:
        json.dump(data, f, indent=4)

# --- FUNGSI CRUD ---
def tampilkan_tabel():
    data = muat_data()
    if not data:
        return pd.DataFrame(columns=["ID", "Nama Produk", "Brand", "Kategori", "Tipe Kulit", "Harga", "Stok", "Deskripsi"])
    df = pd.DataFrame(data)
    df["Harga"] = df["Harga"].apply(lambda x: f"Rp {x:,.0f}".replace(",", "."))
    return df

def dapatkan_pilihan_produk():
    data = muat_data()
    return [f"{p['ID']} - {p['Nama Produk']} ({p['Brand']})" for p in data]

def tambah_produk(nama, brand, kategori, tipe_kulit, harga, stok, deskripsi):
    data = muat_data()
    id_baru = max([p["ID"] for p in data], default=0) + 1

    produk_baru = {
        "ID": id_baru,
        "Nama Produk": nama,
        "Brand": brand,
        "Kategori": kategori,
        "Tipe Kulit": tipe_kulit,
        "Harga": float(harga) if harga else 0,
        "Stok": int(stok) if stok else 0,
        "Deskripsi": deskripsi
    }

    data.append(produk_baru)
    simpan_data(data)

    pilihan_baru = dapatkan_pilihan_produk()
    return (
        tampilkan_tabel(),
        "✅ Produk berhasil ditambahkan dan tersimpan di Google Drive!",
        gr.Dropdown(choices=pilihan_baru, value=pilihan_baru[-1] if pilihan_baru else None),
        gr.Dropdown(choices=pilihan_baru, value=pilihan_baru[0] if pilihan_baru else None)
    )

def isidat_edit(pilihan):
    if not pilihan:
        return "", "", "Cleanser", "Semua Tipe", 0, 0, ""

    data = muat_data()
    id_pilihan = int(pilihan.split(" - ")[0])

    for p in data:
        if p["ID"] == id_pilihan:
            return (
                p["Nama Produk"],
                p["Brand"],
                p["Kategori"],
                p["Tipe Kulit"],
                p["Harga"],
                p["Stok"],
                p.get("Deskripsi", "")
            )
    return "", "", "Cleanser", "Semua Tipe", 0, 0, ""

def update_produk(pilihan, nama, brand, kategori, tipe_kulit, harga, stok, deskripsi):
    if not pilihan:
        return tampilkan_tabel(), "⚠️ Silakan pilih produk yang ingin diubah!"

    data = muat_data()
    id_pilihan = int(pilihan.split(" - ")[0])

    berhasil = False
    for p in data:
        if p["ID"] == id_pilihan:
            p["Nama Produk"] = nama
            p["Brand"] = brand
            p["Kategori"] = kategori
            p["Tipe Kulit"] = tipe_kulit
            p["Harga"] = float(harga)
            p["Stok"] = int(stok)
            p["Deskripsi"] = deskripsi
            berhasil = True
            break

    if berhasil:
        simpan_data(data)
        pilihan_baru = dapatkan_pilihan_produk()
        return (
            tampilkan_tabel(),
            "✅ Data produk berhasil diperbarui di Google Drive!",
            gr.Dropdown(choices=pilihan_baru, value=pilihan),
            gr.Dropdown(choices=pilihan_baru)
        )
    return tampilkan_tabel(), "❌ Gagal memperbarui data.", gr.Dropdown(), gr.Dropdown()

def hapus_produk(pilihan):
    if not pilihan:
        return tampilkan_tabel(), "⚠️ Pilih produk yang ingin dihapus!", gr.Dropdown(), gr.Dropdown()

    data = muat_data()
    id_pilihan = int(pilihan.split(" - ")[0])

    data_baru = [p for p in data if p["ID"] != id_pilihan]
    simpan_data(data_baru)

    pilihan_baru = dapatkan_pilihan_produk()
    pilihan_pertama = pilihan_baru[0] if pilihan_baru else None

    return (
        tampilkan_tabel(),
        "🗑️ Produk berhasil dihapus dari Google Drive!",
        gr.Dropdown(choices=pilihan_baru, value=pilihan_pertama),
        gr.Dropdown(choices=pilihan_baru, value=pilihan_pertama)
    )

# --- DESAIN ANTARMUKA GRADIO ---
theme = gr.themes.Soft(
    primary_hue="sky",
    secondary_hue="indigo",
).set(
    body_background_fill="#F0F8FF",
    block_title_text_color="#1E3A8A"
)

with gr.Blocks(theme=theme, title="GlowCare - Manajemen Skincare") as app:
    gr.Markdown(
        """
        # 🧴 GlowCare - Management Skincare
        ### Aplikasi Manajemen Produk & Stok Skincare (Tersimpan Permanen di Google Drive)
        """
    )

    data_pilihan_awal = dapatkan_pilihan_produk()

    with gr.Tabs():
        # TAB 1: DAFTAR PRODUK
        with gr.TabItem("📋 Daftar Produk"):
            btn_refresh = gr.Button("🔄 Refresh Data", variant="secondary")
            tabel_produk = gr.DataFrame(value=tampilkan_tabel, interactive=False)

        # TAB 2: TAMBAH PRODUK
        with gr.TabItem("➕ Tambah Produk"):
            with gr.Row():
                input_nama = gr.Textbox(label="Nama Produk", placeholder="Contoh: Brightening Serum")
                input_brand = gr.Textbox(label="Brand / Merek", placeholder="Contoh: Purbasari, Wardah")
            with gr.Row():
                input_kat = gr.Dropdown(["Cleanser", "Toner", "Serum", "Moisturizer", "Sunscreen", "Masker"], label="Kategori", value="Serum")
                input_kulit = gr.Dropdown(["Semua Tipe", "Kering", "Berminyak", "Sensitif", "Kombinasi"], label="Tipe Kulit", value="Semua Tipe")
            with gr.Row():
                input_harga = gr.Number(label="Harga (Rp)", value=50000)
                input_stok = gr.Number(label="Jumlah Stok", value=10)
            input_desk = gr.Textbox(label="Deskripsi Produk", lines=2)

            btn_tambah = gr.Button("➕ Simpan Produk Baru", variant="primary")
            status_tambah = gr.Textbox(label="Status", interactive=False)

        # TAB 3: EDIT / UBAH PRODUK
        with gr.TabItem("✏️ Edit / Ubah Produk"):
            dropdown_edit = gr.Dropdown(choices=data_pilihan_awal, label="Pilih Produk yang Ingin Diubah")

            with gr.Row():
                edit_nama = gr.Textbox(label="Nama Produk")
                edit_brand = gr.Textbox(label="Brand / Merek")
            with gr.Row():
                edit_kat = gr.Dropdown(["Cleanser", "Toner", "Serum", "Moisturizer", "Sunscreen", "Masker"], label="Kategori")
                edit_kulit = gr.Dropdown(["Semua Tipe", "Kering", "Berminyak", "Sensitif", "Kombinasi"], label="Tipe Kulit")
            with gr.Row():
                edit_harga = gr.Number(label="Harga (Rp)")
                edit_stok = gr.Number(label="Jumlah Stok")
            edit_desk = gr.Textbox(label="Deskripsi Produk", lines=2)

            btn_update = gr.Button("💾 Simpan Perubahan (Update)", variant="primary")
            status_edit = gr.Textbox(label="Status", interactive=False)

        # TAB 4: HAPUS PRODUK
        with gr.TabItem("🗑️ Hapus Produk"):
            dropdown_hapus = gr.Dropdown(choices=data_pilihan_awal, label="Pilih Produk yang Ingin Dihapus")
            btn_hapus = gr.Button("🗑️ Hapus Produk Ini", variant="stop")
            status_hapus = gr.Textbox(label="Status", interactive=False)

    # --- EVENT HANDLERS ---
    btn_refresh.click(fn=tampilkan_tabel, outputs=tabel_produk)

    dropdown_edit.change(
        fn=isidat_edit,
        inputs=dropdown_edit,
        outputs=[edit_nama, edit_brand, edit_kat, edit_kulit, edit_harga, edit_stok, edit_desk]
    )

    btn_tambah.click(
        fn=tambah_produk,
        inputs=[input_nama, input_brand, input_kat, input_kulit, input_harga, input_stok, input_desk],
        outputs=[tabel_produk, status_tambah, dropdown_edit, dropdown_hapus]
    )

    btn_update.click(
        fn=update_produk,
        inputs=[dropdown_edit, edit_nama, edit_brand, edit_kat, edit_kulit, edit_harga, edit_stok, edit_desk],
        outputs=[tabel_produk, status_edit, dropdown_edit, dropdown_hapus]
    )

    btn_hapus.click(
        fn=hapus_produk,
        inputs=dropdown_hapus,
        outputs=[tabel_produk, status_hapus, dropdown_edit, dropdown_hapus]
    )

# Jalankan Aplikasi
app.launch(share=True, debug=True)

Mounted at /content/drive


/tmp/ipykernel_3368/3216233277.py:185: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="GlowCare - Manajemen Skincare") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://527db67a079f53a3cb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
